# ETAPA 3 — Modelos preentrenados (Transformer — RoBERTa zero-shot)

## Nota importante sobre esta etapa — Uso en modo zero-shot

En esta etapa **no se realizó entrenamiento adicional** del modelo Transformer. Se utilizó el modelo preentrenado
`cardiffnlp/twitter-roberta-base-sentiment` **en modo inferencia directa (zero-shot)** sobre el dataset de reseñas
de Amazon Fine Food, con el objetivo de evaluar la capacidad de **transferencia directa** de un modelo entrenado
en otro dominio (tweets) al dominio de reseñas de productos.

Esto implica:
- No hay fase de entrenamiento ni fine-tuning en este notebook.
- El modelo se aplica tal como fue preentrenado por sus autores.
- El objetivo es cuantificar el desempeño baseline del modelo preentrenado antes de considerar adaptación mediante fine-tuning (Etapa 5).


In [ ]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/notebooks/rodrigolopez29/1-eda/train_etapa1.csv")
val_df   = pd.read_csv("/kaggle/input/notebooks/rodrigolopez29/1-eda/val_etapa1.csv")
test_df  = pd.read_csv("/kaggle/input/notebooks/rodrigolopez29/1-eda/test_etapa1.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# IMPORTANTE: mantenemos los splits SEPARADOS durante toda la etapa.
# Esto evita tener que reconstruirlos por posición (.iloc) más adelante,
# lo cual sería frágil si cambiara el orden de las filas.

print("\nDistribucion clases (train):")
print(train_df["sentiment"].value_counts(normalize=True))
print("\nDistribucion clases (val):")
print(val_df["sentiment"].value_counts(normalize=True))
print("\nDistribucion clases (test):")
print(test_df["sentiment"].value_counts(normalize=True))

train_df.head()

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import pandas as pd
import re
import torch
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

### Justificación de la elección del modelo

El problema requiere **clasificación multiclase** (negative / neutral / positive). Por ello se descartaron modelos
preentrenados únicamente binarios (positivo/negativo) y se seleccionó `cardiffnlp/twitter-roberta-base-sentiment`,
que es un Transformer basado en RoBERTa preentrenado explícitamente para tres clases de sentimiento
(`LABEL_0` = negative, `LABEL_1` = neutral, `LABEL_2` = positive).

La elección responde a dos criterios:

1. **Alineación con el problema**: mapeo directo de las tres clases sin necesidad de post-procesamiento artificial.
2. **Disponibilidad pública y tamaño razonable**: ~125M parámetros, ejecutable en GPU estándar.

La limitación principal es que el modelo fue preentrenado sobre **tweets**, no sobre reseñas de alimentos, por lo
que se espera cierta degradación por *domain shift* que cuantificaremos con las métricas al final de la sección.


A continuación se verifica el entorno y se carga el pipeline de inferencia.

In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))

In [ ]:
from transformers import pipeline
import torch

sentiment_pipeline_multi = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=0 if torch.cuda.is_available() else -1
)

print("Pipeline multiclase listo")

In [ ]:
def predecir_split(df_split, nombre_split, chunk_size=5000, batch_size=32):
    """Aplica el pipeline de sentimiento a un split completo, manteniendo
    el dataframe del split intacto (no concatenamos ni reconstruimos por posición)."""
    textos = df_split["Text_clean"].astype(str).tolist()

    labels = []
    scores = []

    for i in range(0, len(textos), chunk_size):
        chunk = textos[i:i + chunk_size]

        preds = sentiment_pipeline_multi(
            chunk,
            batch_size=batch_size,
            truncation=True,
            max_length=512
        )

        labels.extend([p["label"] for p in preds])
        scores.extend([p["score"] for p in preds])

        print(f"[{nombre_split}] procesado: {min(i + chunk_size, len(textos))} / {len(textos)}")

    df_out = df_split.copy()
    df_out["bert_multi_label"] = labels
    df_out["bert_multi_score"] = scores
    return df_out

# Ejecutamos por cada split por separado
df_train_pred = predecir_split(train_df, "train")
df_val_pred   = predecir_split(val_df,   "val")
df_test_pred  = predecir_split(test_df,  "test")

print("\nShapes con predicciones:")
print("Train:", df_train_pred.shape)
print("Val:  ", df_val_pred.shape)
print("Test: ", df_test_pred.shape)

In [ ]:
def map_multi_label(label):
    if label == "LABEL_0":
        return "negative"
    elif label == "LABEL_1":
        return "neutral"
    else:
        return "positive"

for df_split in (df_train_pred, df_val_pred, df_test_pred):
    df_split["bert_multi_sentiment"] = df_split["bert_multi_label"].apply(map_multi_label)

df_val_pred[["sentiment", "bert_multi_sentiment", "bert_multi_score"]].head()

In [ ]:
# Los splits ya están separados desde la Celda 1; no hace falta reconstruir por posición.
# Aquí solo verificamos tamaños para documentar.
print("Train pred:", df_train_pred.shape)
print("Val pred:  ", df_val_pred.shape)
print("Test pred: ", df_test_pred.shape)
print("Suma:      ", len(df_train_pred) + len(df_val_pred) + len(df_test_pred))

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

y_val_true = df_val_pred["sentiment"]
y_val_pred = df_val_pred["bert_multi_sentiment"]

y_test_true = df_test_pred["sentiment"]
y_test_pred = df_test_pred["bert_multi_sentiment"]

print("Accuracy VALIDACION:", accuracy_score(y_val_true, y_val_pred))
print("F1 macro VALIDACION:", f1_score(y_val_true, y_val_pred, average="macro"))

print("\nClassification report VALIDACION:")
print(classification_report(y_val_true, y_val_pred))

print("\nAccuracy TEST:", accuracy_score(y_test_true, y_test_pred))
print("F1 macro TEST:", f1_score(y_test_true, y_test_pred, average="macro"))

print("\nClassification report TEST:")
print(classification_report(y_test_true, y_test_pred))

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

resumen_transformer = pd.DataFrame({
    "Metrica": [
        "Accuracy",
        "Precision macro",
        "Recall macro",
        "F1 macro",
        "Precision weighted",
        "Recall weighted",
        "F1 weighted"
    ],
    "Validacion": [
        accuracy_score(y_val_true, y_val_pred),
        precision_score(y_val_true, y_val_pred, average="macro"),
        recall_score(y_val_true, y_val_pred, average="macro"),
        f1_score(y_val_true, y_val_pred, average="macro"),
        precision_score(y_val_true, y_val_pred, average="weighted"),
        recall_score(y_val_true, y_val_pred, average="weighted"),
        f1_score(y_val_true, y_val_pred, average="weighted")
    ],
    "Test": [
        accuracy_score(y_test_true, y_test_pred),
        precision_score(y_test_true, y_test_pred, average="macro"),
        recall_score(y_test_true, y_test_pred, average="macro"),
        f1_score(y_test_true, y_test_pred, average="macro"),
        precision_score(y_test_true, y_test_pred, average="weighted"),
        recall_score(y_test_true, y_test_pred, average="weighted"),
        f1_score(y_test_true, y_test_pred, average="weighted")
    ]
})

resumen_transformer["Modelo"] = "Transformer"
resumen_transformer = resumen_transformer[["Modelo", "Metrica", "Validacion", "Test"]]
resumen_transformer.to_csv("transformer_metrics.csv", index=False)

resumen_transformer

In [3]:
import pandas as pd                                                                                                                                                                                           
                                                                                                                                                                                                        
# ========================================================
# Métricas calculadas en la corrida original (v1, 3.5h).                                                                                                                                                      
# La inferencia ya se ejecutó; guardamos el CSV directamente
# para que quede disponible como output de esta versión.                                                                                                                                                      
# ========================================================
                                                                                                                                                                                                        
resumen_transformer = pd.DataFrame({                                                                                                                                                                          
"Modelo": ["Transformer"] * 7,                                                                                                                                                                            
"Metrica": [                                                                                                                                                                                              
  "Accuracy", "Precision macro", "Recall macro", "F1 macro",                                                                                                                                            
  "Precision weighted", "Recall weighted", "F1 weighted"
],                                                                                                                                                                                                        
"Validacion": [0.833812, 0.614692, 0.618255, 0.616082, 0.841413, 0.833812, 0.837499],
"Test":       [0.834440, 0.615702, 0.618163, 0.616629, 0.840882, 0.834440, 0.837575]                                                                                                                      
})                                          
resumen_transformer = resumen_transformer[["Modelo", "Metrica", "Validacion", "Test"]]                                                                                                                        
resumen_transformer.to_csv("transformer_metrics.csv", index=False)                                                                                                                                            
                                                                                                                                                                                                        
print("CSV regenerado:")                                                                                                                                                                                      
print(resumen_transformer)                                                 

CSV regenerado:
        Modelo             Metrica  Validacion      Test
0  Transformer            Accuracy    0.833812  0.834440
1  Transformer     Precision macro    0.614692  0.615702
2  Transformer        Recall macro    0.618255  0.618163
3  Transformer            F1 macro    0.616082  0.616629
4  Transformer  Precision weighted    0.841413  0.840882
5  Transformer     Recall weighted    0.833812  0.834440
6  Transformer         F1 weighted    0.837499  0.837575
